# Meyaar Vision: Moondream Element-Presence Fine-Tuning

This notebook trains and evaluates a Moondream model that answers whether four essential cartographic elements are present in a map image: **title**, **legend**, **scale**, and **north arrow**.

The workflow uses controlled error injection. Original maps provide positive examples, while element regions are removed with inpainting to create paired negative examples. Original maps are split before injection to prevent data leakage.

Moondream's current official fine-tuning service performs training in Moondream Cloud. Colab is used here for dataset preparation, orchestration, evaluation, and artifact storage; a Colab GPU is not required.

## 1. Install dependencies

Restart the Colab runtime if an installation message asks you to do so.

In [7]:
!pip -q install -U moondream iterative-stratification opencv-python-headless scikit-learn pandas pillow matplotlib

## 2. Mount Google Drive and configure paths

Add a shortcut for `maps (2).zip` to the root of My Drive before running this section. Generated images are stored temporarily in the Colab runtime. Persistent CSV files and model metadata are saved to Google Drive.

In [8]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_PROJECT = Path('/content/drive/MyDrive/Meyaar')
ZIP_PATH = Path('/content/drive/MyDrive/maps (2).zip')
LOCAL_DATA = Path('/content/meyaar_maps')
IMAGES_PATH = LOCAL_DATA / 'maps'
LABELS_PATH = LOCAL_DATA / 'labels'
INJECTED_PATH = Path('/content/meyaar_injected')
ARTIFACTS_PATH = DRIVE_PROJECT / 'moondream_artifacts'

DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
INJECTED_PATH.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.exists(), f'ZIP file not found: {ZIP_PATH}'
print('ZIP:', ZIP_PATH)

Mounted at /content/drive
ZIP: /content/drive/MyDrive/maps (2).zip


## 3. Extract and validate the dataset

In [9]:
import shutil

if not IMAGES_PATH.exists() or not LABELS_PATH.exists():
    shutil.unpack_archive(str(ZIP_PATH), str(LOCAL_DATA))

image_files = sorted([p for p in IMAGES_PATH.iterdir() if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}])
label_files = sorted(LABELS_PATH.glob('*.json'))

print('Images:', len(image_files))
print('Labels:', len(label_files))
assert len(image_files) == 703
assert len(label_files) == 703

Images: 703
Labels: 703


## 4. Parse annotations and build the original-map table

Related raw annotation classes are merged into four target elements. Bounding boxes are retained for controlled removal.

In [10]:
import json
import pandas as pd
from PIL import Image

ELEMENT_SOURCES = {
    'title': {'title'},
    'legend': {'legend-color', 'legend-symbol', 'legend-mixed'},
    'scale': {'scale-graphic', 'scale-numeric'},
    'north_arrow': {'orient-arrow'},
}
TARGET_ELEMENTS = list(ELEMENT_SOURCES)

def annotation_to_bbox(annotation):
    points = annotation.get('obj_points', [])
    if not points:
        return None
    if annotation.get('obj_type') == 1:
        point = points[0]
        if not all(key in point for key in ('x', 'y', 'w', 'h')):
            return None
        x1, y1 = float(point['x']), float(point['y'])
        x2, y2 = x1 + float(point['w']), y1 + float(point['h'])
    else:
        valid = [point for point in points if 'x' in point and 'y' in point]
        if not valid:
            return None
        xs = [float(point['x']) for point in valid]
        ys = [float(point['y']) for point in valid]
        x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
    return x1, y1, x2, y2

def clip_bbox(bbox, width, height):
    if bbox is None:
        return None
    x1, y1, x2, y2 = bbox
    x1 = max(0, min(width - 1, int(round(x1))))
    y1 = max(0, min(height - 1, int(round(y1))))
    x2 = max(x1 + 1, min(width, int(round(x2))))
    y2 = max(y1 + 1, min(height, int(round(y2))))
    return x1, y1, x2, y2

records = []
for label_path in label_files:
    image_name = label_path.name.removesuffix('.json')
    image_path = IMAGES_PATH / image_name
    if not image_path.exists():
        continue
    with Image.open(image_path) as image:
        width, height = image.size
    with open(label_path, 'r', encoding='utf-8') as file:
        annotations = json.load(file)
    element_boxes = {element: [] for element in TARGET_ELEMENTS}
    for annotation in annotations:
        raw_name = annotation.get('f_name')
        for element, source_names in ELEMENT_SOURCES.items():
            if raw_name in source_names:
                bbox = clip_bbox(annotation_to_bbox(annotation), width, height)
                if bbox is not None:
                    element_boxes[element].append(bbox)
    record = {
        'image_name': image_name,
        'image_path': str(image_path),
        'width': width,
        'height': height,
        'element_boxes': element_boxes,
    }
    for element in TARGET_ELEMENTS:
        record[element] = int(bool(element_boxes[element]))
    records.append(record)

original_dataset = pd.DataFrame(records)
print('Original maps:', len(original_dataset))
display(original_dataset[TARGET_ELEMENTS].sum().rename('present_count').to_frame())
assert len(original_dataset) == 703

Original maps: 703


,present_count
title,394
legend,658
scale,476
north_arrow,253


## 5. Split original maps before injection

Multilabel stratification keeps the four element-presence rates similar across train, validation, and test. The test set remains untouched until the final evaluation.

In [13]:
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

SEED = 42
y = original_dataset[TARGET_ELEMENTS].to_numpy()
x = np.zeros((len(original_dataset), 1))
first_split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_indices, temporary_indices = next(first_split.split(x, y))
temporary_y = y[temporary_indices]
temporary_x = np.zeros((len(temporary_indices), 1))
second_split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
validation_positions, test_positions = next(second_split.split(temporary_x, temporary_y))
validation_indices = temporary_indices[validation_positions]
test_indices = temporary_indices[test_positions]

original_dataset['split'] = ''
original_dataset.loc[train_indices, 'split'] = 'train'
original_dataset.loc[validation_indices, 'split'] = 'validation'
original_dataset.loc[test_indices, 'split'] = 'test'

assert (original_dataset['split'] != '').all()
assert original_dataset['image_name'].nunique() == len(original_dataset)
print(original_dataset['split'].value_counts())
display(original_dataset.groupby('split')[TARGET_ELEMENTS].mean())
original_dataset.drop(columns=['element_boxes']).to_csv(ARTIFACTS_PATH / 'original_split.csv', index=False)

split
train         492
test          106
validation    105
Name: count, dtype: int64


,title,legend,scale,north_arrow
split,,,,
test,0.556604,0.924528,0.679245,0.358491
train,0.560976,0.936992,0.676829,0.359756
validation,0.561905,0.942857,0.676190,0.361905


## 6. Build Natural Element-Presence Dataset

Each original map produces four question-answer samples: title, legend, scale, and north arrow. Targets come directly from the original JSON annotations. Controlled error injection is excluded from training and will be retained only as a separate experimental benchmark after manual quality review.

In [27]:
ELEMENT_QUESTIONS = {
    "title": (
        "Is the map title present in this map? "
        "Answer only yes or no."
    ),
    "legend": (
        "Is the map legend present in this map? "
        "Answer only yes or no."
    ),
    "scale": (
        "Is the map scale present in this map? "
        "Answer only yes or no."
    ),
    "north_arrow": (
        "Is the north arrow present in this map? "
        "Answer only yes or no."
    ),
}

natural_rows = []

for row in original_dataset.itertuples():
    for element in TARGET_ELEMENTS:
        natural_rows.append({
            "source_image": row.image_name,
            "image_path": row.image_path,
            "split": row.split,
            "element": element,
            "question": ELEMENT_QUESTIONS[element],
            "target": (
                "yes"
                if getattr(row, element) == 1
                else "no"
            ),
            "is_injected": False,
        })

natural_dataset = pd.DataFrame(natural_rows)

train_frame = natural_dataset[
    natural_dataset["split"] == "train"
].copy()

validation_frame = natural_dataset[
    natural_dataset["split"] == "validation"
].copy()

test_frame = natural_dataset[
    natural_dataset["split"] == "test"
].copy()

print("Natural samples:", len(natural_dataset))
print("Train:", len(train_frame))
print("Validation:", len(validation_frame))
print("Test:", len(test_frame))

display(
    natural_dataset.groupby(
        ["split", "element", "target"]
    ).size().rename("count").to_frame()
)

Natural samples: 2812
Train: 1968
Validation: 420
Test: 424


count
split      element     target       
test       legend      no          8
                       yes        98
           north_arrow no         68
                       yes        38
           scale       no         34
                       yes        72
           title       no         47
                       yes        59
train      legend      no         31
                       yes       461
           north_arrow no        315
                       yes       177
           scale       no        159
                       yes       333
           title       no        216
                       yes       276
validation legend      no          6
                       yes        99
           north_arrow no         67
                       yes        38
           scale       no         34
                       yes        71
           title       no         46
                       yes        59

## 7. Balance Natural Training Data

The training set is balanced across element and target combinations. Validation and test sets remain unchanged to represent the natural data distribution.

In [48]:
SAMPLES_PER_GROUP = 100

balanced_groups = []

for _, group in train_frame.groupby(
    ["element", "target"]
):
    balanced_groups.append(
        group.sample(
            n=SAMPLES_PER_GROUP,
            replace=len(group) < SAMPLES_PER_GROUP,
            random_state=SEED,
        )
    )

train_balanced = (
    pd.concat(
        balanced_groups,
        ignore_index=True,
    )
    .sample(
        frac=1,
        random_state=SEED,
    )
    .reset_index(drop=True)
)

print("Training samples:", len(train_balanced))

print(
    train_balanced.groupby(
        ["element", "target"]
    ).size()
)

Training samples: 800
element      target
legend       no        100
             yes       100
north_arrow  no        100
             yes       100
scale        no        100
             yes       100
title        no        100
             yes       100
dtype: int64


## 8. Connect to Moondream

The API key is loaded from Colab Secrets. It must never be written directly in the notebook or uploaded to GitHub.

In [49]:
from google.colab import userdata
import moondream as md


MOONDREAM_API_KEY = userdata.get(
    "Moondream"
)

assert MOONDREAM_API_KEY, (
    "MOONDREAM_API_KEY is missing "
    "from Colab Secrets."
)

base_model = md.vl(
    api_key=MOONDREAM_API_KEY
)

print("Moondream connected")

Moondream connected


## 9. Evaluation Functions

Images are resized before being sent to Moondream. Temporary server errors are retried. Invalid responses are counted and reported.

In [50]:
import time

from urllib.error import HTTPError

import pandas as pd

from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
)


def normalize_answer(value):
    answer = str(value).strip().lower()

    if answer.startswith("yes"):
        return "yes"

    if answer.startswith("no"):
        return "no"

    return "invalid"


def query_model(
    model,
    image_path,
    question,
    max_attempts=5,
):
    with Image.open(image_path) as source:
        image = source.convert("RGB")

        image.thumbnail(
            (1024, 1024),
            Image.Resampling.LANCZOS,
        )

    try:
        for attempt in range(max_attempts):
            try:
                result = model.query(
                    image,
                    question,
                    settings={
                        "temperature": 0.0,
                        "max_tokens": 4,
                    },
                )

                return normalize_answer(
                    result["answer"]
                )

            except HTTPError as error:
                if attempt == max_attempts - 1:
                    print(
                        f"Failed: {image_path}, "
                        f"HTTP {error.code}"
                    )

                    return "invalid"

                wait_seconds = 10 * (attempt + 1)

                print(
                    f"HTTP {error.code}. "
                    f"Retrying in {wait_seconds}s"
                )

                time.sleep(wait_seconds)

    finally:
        image.close()


def select_balanced_evaluation_sample(
    frame,
    limit,
):
    if limit is None or len(frame) <= limit:
        return (
            frame.copy()
            .reset_index(drop=True)
        )

    grouped = frame.groupby(
        ["element", "target"]
    )

    group_count = grouped.ngroups
    per_group = max(
        1,
        limit // group_count,
    )

    sampled_groups = []

    for _, group in grouped:
        sample_size = min(
            len(group),
            per_group,
        )

        sampled_groups.append(
            group.sample(
                n=sample_size,
                random_state=SEED,
            )
        )

    return pd.concat(
        sampled_groups,
        ignore_index=True,
    )


def calculate_metrics(results):
    accuracy = accuracy_score(
        results["target"],
        results["prediction"],
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            results["target"],
            results["prediction"],
            labels=["yes", "no"],
            average="macro",
            zero_division=0,
        )
    )

    valid_predictions = int(
        results["prediction"]
        .isin(["yes", "no"])
        .sum()
    )

    return {
        "samples": len(results),
        "valid_predictions": valid_predictions,
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }


def evaluate_model(
    model,
    frame,
    limit=None,
):
    evaluation = (
        select_balanced_evaluation_sample(
            frame,
            limit,
        )
    )

    predictions = []

    for position, row in evaluation.iterrows():
        prediction = query_model(
            model,
            row["image_path"],
            row["question"],
        )

        predictions.append(prediction)

        completed = position + 1

        if (
            completed % 25 == 0
            or completed == len(evaluation)
        ):
            print(
                f"{completed}/{len(evaluation)}"
            )

    evaluation["prediction"] = predictions

    metrics = calculate_metrics(
        evaluation
    )

    return metrics, evaluation


def calculate_element_metrics(results):
    metric_rows = []

    for element, group in results.groupby(
        "element"
    ):
        precision, recall, f1, _ = (
            precision_recall_fscore_support(
                group["target"],
                group["prediction"],
                labels=["yes", "no"],
                average="macro",
                zero_division=0,
            )
        )

        valid_predictions = int(
            group["prediction"]
            .isin(["yes", "no"])
            .sum()
        )

        metric_rows.append({
            "element": element,
            "samples": len(group),
            "valid_predictions": valid_predictions,
            "accuracy": accuracy_score(
                group["target"],
                group["prediction"],
            ),
            "macro_precision": precision,
            "macro_recall": recall,
            "macro_f1": f1,
        })

    return pd.DataFrame(
        metric_rows
    )

In [51]:
def calculate_metrics(results):
    accuracy = accuracy_score(
        results["target"],
        results["prediction"],
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            results["target"],
            results["prediction"],
            labels=["yes", "no"],
            average="macro",
            zero_division=0,
        )
    )

    return {
        "samples": len(results),
        "valid_predictions": int(
            results["prediction"]
            .isin(["yes", "no"])
            .sum()
        ),
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }

## 10. Evaluate the Base Model

A fixed balanced validation sample is evaluated before training. The same sample will be used after fine-tuning for a fair comparison.

In [52]:
VALIDATION_LIMIT = 80

baseline_metrics, baseline_results = (
    evaluate_model(
        base_model,
        validation_frame,
        VALIDATION_LIMIT,
    )
)

print(baseline_metrics)

display(
    calculate_element_metrics(
        baseline_results
    )
)

baseline_results.to_csv(
    ARTIFACTS_PATH
    / "natural_baseline_predictions.csv",
    index=False,
)

25/76
50/76
75/76
76/76
{'samples': 76, 'valid_predictions': 76, 'accuracy': 0.8157894736842105, 'macro_precision': 0.8318452380952381, 'macro_recall': 0.8097222222222222, 'macro_f1': 0.8110795454545454}


,element,samples,valid_predictions,accuracy,macro_precision,macro_recall,macro_f1
0,legend,16,16,0.8125,0.884615,0.75,0.768116
1,north_arrow,20,20,0.9000,0.916667,0.90,0.898990
2,scale,20,20,0.8500,0.884615,0.85,0.846547
3,title,20,20,0.7000,0.700000,0.70,0.700000


## 11. Create a Natural-Data Fine-Tune

This training run uses only natural map images and labels derived from the original JSON annotations. Set START_TRAINING to True only after reviewing the dataset and baseline.

In [53]:
import math
import time

START_TRAINING = True

assert START_TRAINING, (
    "Change START_TRAINING to True "
    "when ready."
)

FINETUNE_NAME = (
    f"meyaar-balanced-800-lr2e5-{int(time.time())}"
)

LORA_RANK = 8
LEARNING_RATE = 2e-5
BATCH_SIZE = 4
EPOCHS = 1

finetune = md.ft(
    api_key=MOONDREAM_API_KEY,
    name=FINETUNE_NAME,
    rank=LORA_RANK,
)

FINETUNE_ID = finetune.finetune_id

print("Finetune ID:", FINETUNE_ID)
print("Training samples:", len(train_balanced))

print(
    "Training batches:",
    math.ceil(
        len(train_balanced) / BATCH_SIZE
    ),
)

finetune_state = {
    "finetune_id": FINETUNE_ID,
    "name": FINETUNE_NAME,
    "rank": LORA_RANK,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "training_samples": len(train_balanced),
}

with open(
    ARTIFACTS_PATH
    / "natural_finetune_state.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        finetune_state,
        file,
        indent=2,
    )

Finetune ID: 01M1PM29FRHRPMRS7JE58SZPF9
Training samples: 800
Training batches: 200


## 12. Run Supervised Fine-Tuning

Each training request contains a natural map image, an element-presence question, and the expected yes-or-no answer. Checkpoints are saved periodically.

In [54]:
def load_training_image(
    image_path,
    max_size=(1024, 1024),
):
    with Image.open(image_path) as source:
        image = source.convert("RGB")

        image.thumbnail(
            max_size,
            Image.Resampling.LANCZOS,
        )

        return image.copy()

In [55]:
SAMPLES_PER_GROUP = 100

balanced_groups = []

for _, group in train_frame.groupby(
    ["element", "target"]
):
    balanced_groups.append(
        group.sample(
            n=SAMPLES_PER_GROUP,
            replace=len(group) < SAMPLES_PER_GROUP,
            random_state=SEED,
        )
    )

train_balanced = (
    pd.concat(
        balanced_groups,
        ignore_index=True,
    )
    .sample(
        frac=1,
        random_state=SEED,
    )
    .reset_index(drop=True)
)

print("Training samples:", len(train_balanced))

print(
    train_balanced.groupby(
        ["element", "target"]
    ).size()
)

Training samples: 800
element      target
legend       no        100
             yes       100
north_arrow  no        100
             yes       100
scale        no        100
             yes       100
title        no        100
             yes       100
dtype: int64


In [56]:
training_history = []
successful_batches = 0

for epoch in range(EPOCHS):
    epoch_frame = train_balanced.sample(
        frac=1,
        random_state=SEED + epoch,
    ).reset_index(drop=True)

    for start in range(
        0,
        len(epoch_frame),
        BATCH_SIZE,
    ):
        batch = epoch_frame.iloc[
            start:start + BATCH_SIZE
        ]

        groups = []
        opened_images = []

        try:
            for row in batch.itertuples():
                image = load_training_image(
                    row.image_path
                )

                opened_images.append(image)

                groups.append({
                    "mode": "sft",
                    "request": {
                        "skill": "query",
                        "image": image,
                        "question": row.question,
                    },
                    "target": {
                        "answer": row.target,
                    },
                })

            result = finetune.train_step(
                groups,
                lr=LEARNING_RATE,
            )

            training_history.append(result)
            successful_batches += 1

        except HTTPError as error:
            print(
                "Training stopped at "
                f"sample {start}."
            )

            print(
                "HTTP status:",
                error.code,
            )

            print(
                "Check the latest server "
                "checkpoint before resuming."
            )

            raise

        finally:
            for image in opened_images:
                image.close()

        completed = min(
            start + BATCH_SIZE,
            len(epoch_frame),
        )

        if (
            completed % 40 == 0
            or completed == len(epoch_frame)
        ):
            print(
                f"Epoch {epoch + 1}/{EPOCHS}: "
                f"{completed}/{len(epoch_frame)}"
            )

        if successful_batches % 50 == 0:
            saved = (
                finetune.save_checkpoint()
                ["checkpoint"]
            )

            print(
                "Saved checkpoint:",
                saved["step"],
            )

training_history_frame = pd.DataFrame(
    training_history
)

training_history_frame.to_csv(
    ARTIFACTS_PATH
    / "natural_training_history.csv",
    index=False,
)

Epoch 1/1: 40/800
Epoch 1/1: 80/800
Epoch 1/1: 120/800
Epoch 1/1: 160/800
Epoch 1/1: 200/800
Saved checkpoint: 50
Epoch 1/1: 240/800
Epoch 1/1: 280/800
Epoch 1/1: 320/800
Epoch 1/1: 360/800
Epoch 1/1: 400/800
Saved checkpoint: 100
Epoch 1/1: 440/800
Epoch 1/1: 480/800
Epoch 1/1: 520/800
Epoch 1/1: 560/800
Epoch 1/1: 600/800
Saved checkpoint: 150
Epoch 1/1: 640/800
Epoch 1/1: 680/800
Epoch 1/1: 720/800
Epoch 1/1: 760/800
Epoch 1/1: 800/800
Saved checkpoint: 200


## 13. Save the Final Checkpoint and Evaluate

The final checkpoint is evaluated on the same fixed validation sample used for the baseline.

In [57]:
checkpoint_response = (
    finetune.save_checkpoint()
)

checkpoint = checkpoint_response[
    "checkpoint"
]

MODEL_ID = finetune.model(
    checkpoint["step"]
)

print(
    "Checkpoint step:",
    checkpoint["step"],
)

print(
    "Model ID:",
    MODEL_ID,
)

Checkpoint step: 200
Model ID: moondream3-preview/01M1PM29FRHRPMRS7JE58SZPF9@200


In [58]:
time.sleep(30)

tuned_model = md.vl(
    api_key=MOONDREAM_API_KEY,
    model=MODEL_ID,
)

tuned_metrics, tuned_results = (
    evaluate_model(
        tuned_model,
        validation_frame,
        VALIDATION_LIMIT,
    )
)

print("Baseline:")
print(baseline_metrics)

print("Fine-tuned:")
print(tuned_metrics)

display(
    calculate_element_metrics(
        tuned_results
    )
)

tuned_results.to_csv(
    ARTIFACTS_PATH
    / "natural_finetuned_predictions.csv",
    index=False,
)

25/76
50/76
75/76
76/76
Baseline:
{'samples': 76, 'valid_predictions': 76, 'accuracy': 0.8157894736842105, 'macro_precision': 0.8318452380952381, 'macro_recall': 0.8097222222222222, 'macro_f1': 0.8110795454545454}
Fine-tuned:
{'samples': 76, 'valid_predictions': 76, 'accuracy': 0.8289473684210527, 'macro_precision': 0.8418928833455612, 'macro_recall': 0.8236111111111111, 'macro_f1': 0.825287356321839}


,element,samples,valid_predictions,accuracy,macro_precision,macro_recall,macro_f1
0,legend,16,16,0.8125,0.884615,0.75,0.768116
1,north_arrow,20,20,0.9500,0.954545,0.95,0.949875
2,scale,20,20,0.8500,0.884615,0.85,0.846547
3,title,20,20,0.7000,0.700000,0.70,0.700000


In [59]:
comparison = pd.DataFrame([
    {
        "model": "base",
        **baseline_metrics,
    },
    {
        "model": "fine_tuned",
        **tuned_metrics,
    },
])

display(comparison)

comparison.to_csv(
    ARTIFACTS_PATH
    / "model_comparison.csv",
    index=False,
)

,model,samples,valid_predictions,accuracy,macro_precision,macro_recall,macro_f1
0,base,76,76,0.815789,0.831845,0.809722,0.811080
1,fine_tuned,76,76,0.828947,0.841893,0.823611,0.825287


In [60]:
calculate_element_metrics(tuned_results)

,element,samples,valid_predictions,accuracy,macro_precision,macro_recall,macro_f1
0,legend,16,16,0.8125,0.884615,0.75,0.768116
1,north_arrow,20,20,0.9500,0.954545,0.95,0.949875
2,scale,20,20,0.8500,0.884615,0.85,0.846547
3,title,20,20,0.7000,0.700000,0.70,0.700000


## 14. Final Natural Test

Run this section only once after selecting the final model using validation results. The natural test set is the primary project benchmark.

In [62]:
RUN_FINAL_TEST = True

assert RUN_FINAL_TEST, (
    "Change RUN_FINAL_TEST to True "
    "only after selecting the final model."
)

In [65]:
final_test_metrics, final_test_results = (
    evaluate_model(
        tuned_model,
        test_frame,
        limit=None,
    )
)

print(final_test_metrics)

display(
    calculate_element_metrics(
        final_test_results
    )
)

final_test_results.to_csv(
    ARTIFACTS_PATH
    / "natural_test_predictions.csv",
    index=False,
)

with open(
    ARTIFACTS_PATH
    / "natural_test_metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_test_metrics,
        file,
        indent=2,
    )

25/424
50/424
75/424
100/424
125/424
150/424
175/424
200/424
225/424
250/424
275/424
300/424
325/424
350/424
375/424
400/424
424/424
{'samples': 424, 'valid_predictions': 424, 'accuracy': 0.9127358490566038, 'macro_precision': 0.9241905354919053, 'macro_recall': 0.8900379302941387, 'macro_f1': 0.9028913470049706}


,element,samples,valid_predictions,accuracy,macro_precision,macro_recall,macro_f1
0,legend,106,106,0.933962,0.775248,0.677296,0.713181
1,north_arrow,106,106,0.886792,0.876796,0.905960,0.882614
2,scale,106,106,0.886792,0.928571,0.823529,0.854396
3,title,106,106,0.943396,0.942661,0.942661,0.942661


## 15. Save Model Information for Backend Integration

The model ID is safe to store in project configuration. The API key must remain in environment variables or a secret manager.

In [66]:
model_metadata = {
    "finetune_id": FINETUNE_ID,
    "checkpoint_step": checkpoint["step"],
    "model_id": MODEL_ID,
    "training_source": "natural_annotations",
    "baseline_metrics": baseline_metrics,
    "validation_metrics": tuned_metrics,
}

with open(
    ARTIFACTS_PATH
    / "natural_model_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        model_metadata,
        file,
        indent=2,
    )

print(
    "Saved:",
    ARTIFACTS_PATH
    / "natural_model_metadata.json",
)

Saved: /content/drive/MyDrive/Meyaar/moondream_artifacts/natural_model_metadata.json
